# User-Level Feature Table and Label Engineering

Builds the modeling data asset: a one-row-per-user wide table with behavioral, ad, and subscription features aggregated over the observation window, plus churn and paid-conversion labels defined over the prediction window.

> Runs on `pandas + sqlite3`. When running in Colab, upload the five CSV files to `/content/data` before executing the data-loading cell.


In [1]:
# Environment setup: load the project tables into in-memory SQLite.
# See src/data_loader.py — data is auto-discovered in ./data (repo) or
# /content/data (Colab); override with the SPOTIFY_DATA_DIR env variable.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

from src.data_loader import connect

conn, sql, run_script = connect()

/Users/jackyjiang/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/jackyjiang/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


Loaded tables: users, listening_events, subscription_events, ad_events, feature_table


## 1. Input Table Granularity Check

This notebook builds a user-level wide table from the base tables, so first confirm which tables are user-grained and which are event-grained.


In [2]:
query = """SELECT 'users' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM users

UNION ALL

SELECT 'listening_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM listening_events

UNION ALL

SELECT 'subscription_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM subscription_events

UNION ALL

SELECT 'ad_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM ad_events

UNION ALL

SELECT 'feature_table' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM feature_table
;"""

sql(query)

,table_name,row_count,user_count
0,users,50000,50000
1,listening_events,1072063,48187
2,subscription_events,104498,34640
3,ad_events,310438,31000
4,feature_table,50000,50000


## 2. Time Window Design: Snapshot / Observation / Prediction

A wide table is not just concatenated fields: at a given snapshot_date, features are built from past behavior and labels are defined from future outcomes.


In [3]:
query = """SELECT
    '2026-04-01' AS snapshot_date,
    '2026-03-02' AS observation_start,
    '2026-04-01' AS observation_end_exclusive,
    '2026-04-15' AS churn_prediction_end_exclusive,
    '2026-05-01' AS conversion_prediction_end_exclusive
;"""

sql(query)

,snapshot_date,observation_start,observation_end_exclusive,churn_prediction_end_exclusive,conversion_prediction_end_exclusive
0,2026-04-01,2026-03-02,2026-04-01,2026-04-15,2026-05-01


## 3. base_users: Defining the User Pool

The first step of the wide table is defining the user pool. All subsequent features are LEFT JOINed to base_users to guarantee the final table is one row per user.


In [4]:
query = """WITH latest_subscription AS (
    SELECT
        user_id,
        plan_type AS latest_plan_type,
        event_type AS latest_subscription_event
    FROM (
        SELECT
            user_id,
            plan_type,
            event_type,
            event_timestamp,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_timestamp DESC
            ) AS rn
        FROM subscription_events
        WHERE event_timestamp < '2026-04-01'
    )
    WHERE rn = 1
)
SELECT
    u.user_id,
    u.signup_date,
    u.country,
    u.language,
    u.region_tier,
    u.age_group,
    u.primary_device,
    u.acquisition_channel,
    u.student_eligible,
    u.marketing_opt_in,
    u.music_persona,
    COALESCE(l.latest_plan_type, 'free') AS current_subscription_type,
    '2026-04-01' AS snapshot_date
FROM users u
LEFT JOIN latest_subscription l
    ON u.user_id = l.user_id
LIMIT 10
;"""

sql(query)

,user_id,signup_date,country,language,region_tier,age_group,primary_device,acquisition_channel,student_eligible,marketing_opt_in,music_persona,current_subscription_type,snapshot_date
0,U000001,2026-02-06,UK,en,Tier 1,35-44,mobile_android,organic_search,0,0,commute_listener,individual_premium,2026-04-01
1,U000002,2026-03-07,ES,es,Tier 2,25-34,mobile_android,paid_social,0,0,playlist_builder,trial,2026-04-01
2,U000003,2025-07-29,IN,en,Tier 2,18-24,mobile_android,referral,0,0,casual_listener,trial,2026-04-01
3,U000004,2025-11-14,FR,fr,Tier 1,35-44,mobile_android,referral,0,1,commute_listener,trial,2026-04-01
4,U000005,2025-05-21,US,en,Tier 1,25-34,mobile_ios,student_partner,1,0,casual_listener,free,2026-04-01
5,U000006,2026-03-09,US,en,Tier 1,25-34,mobile_android,email_campaign,0,0,commute_listener,trial,2026-04-01
6,U000007,2025-12-17,US,en,Tier 1,18-24,desktop,email_campaign,1,0,playlist_builder,trial,2026-04-01
7,U000008,2026-01-19,MX,es,Tier 2,25-34,desktop,app_store,0,1,social_listener,free,2026-04-01
8,U000009,2026-03-06,FR,fr,Tier 1,18-24,mobile_ios,paid_social,1,1,playlist_builder,trial,2026-04-01
9,U000010,2025-12-21,IN,en,Tier 2,18-24,mobile_android,paid_social,1,0,casual_listener,individual_premium,2026-04-01


## 4. listening_features: Behavioral Features from Listening Events

Churn prediction and retention analysis first need to know whether the user has formed a usage habit and whether they interact with content.


In [5]:
query = """-- build listening_features
SELECT
    user_id,
    Count(Distinct date(event_timestamp)) as active_days_30d,
    count(*) as listen_events_30d,
    Round(sum(play_duration_sec) / 60.0 ,2 ) as listen_min_30d,
    Count(distinct session_id) as sessions_30d,
    Round(avg(play_duration_sec),2) as avg_play_duration_sec_30d,
    Round(avg(skipped_flag), 4 ) as skip_rate_30d,
    Round(avg(completed_flag), 4 ) as completion_rate_30d,
    SUM(liked_flag) as liked_songs_30d,
    SUM(playlist_add_flag) as playlist_adds_30d,
    SUM(search_used_flag) as search_evernts_30d

FROM listening_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 100
;"""

sql(query)


,user_id,active_days_30d,listen_events_30d,listen_min_30d,sessions_30d,avg_play_duration_sec_30d,skip_rate_30d,completion_rate_30d,liked_songs_30d,playlist_adds_30d,search_evernts_30d
0,U000001,1,1,1.93,1,116.00,0.0000,0.0000,0,0,0
1,U000002,10,10,21.55,10,129.30,0.2000,0.3000,1,1,2
2,U000003,3,3,6.50,3,130.00,0.0000,0.0000,0,0,0
3,U000004,4,4,11.33,4,170.00,0.2500,0.0000,0,0,3
4,U000005,1,1,2.72,1,163.00,0.0000,0.0000,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
95,U000108,2,3,47.42,3,948.33,0.3333,0.6667,0,0,0
96,U000109,10,10,53.02,10,318.10,0.1000,0.2000,1,0,3
97,U000111,7,8,18.98,8,142.38,0.1250,0.5000,2,0,4
98,U000112,8,9,92.50,8,616.67,0.1111,0.4444,0,1,1


## 5. ad_features: Ad Pressure and Monetization Features

Ads monetize free users and may also affect experience and retention. Both models and diagnostics need ad load features.


In [6]:
query = """SELECT
    user_id,
    COUNT(*) AS ad_impressions_30d,
    SUM(clicked_flag) AS ad_clicks_30d,
    ROUND(SUM(clicked_flag) * 1.0 / NULLIF(COUNT(*), 0), 4) AS ad_click_rate_30d,
    SUM(completed_flag) AS ad_completions_30d,
    ROUND(SUM(completed_flag) * 1.0 / NULLIF(COUNT(*), 0), 4) AS ad_completion_rate_30d,
    ROUND(SUM(revenue_usd), 4) AS ad_revenue_30d
FROM ad_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 10
;"""

sql(query)

,user_id,ad_impressions_30d,ad_clicks_30d,ad_click_rate_30d,ad_completions_30d,ad_completion_rate_30d,ad_revenue_30d
0,U000001,3,0,0.0,2,0.6667,0.0438
1,U000002,7,0,0.0,6,0.8571,0.0544
2,U000003,1,0,0.0,0,0.0000,0.0025
3,U000004,2,0,0.0,1,0.5000,0.0703
4,U000006,4,0,0.0,2,0.5000,0.0671
5,U000007,16,0,0.0,9,0.5625,0.2380
6,U000008,2,0,0.0,1,0.5000,0.0106
7,U000009,4,2,0.5,3,0.7500,0.1573
8,U000011,1,0,0.0,1,1.0000,0.0096
9,U000012,2,0,0.0,2,1.0000,0.0256


## 6. subscription_features: Trial, Payment, Renewal, and Cancellation Features

Subscription behavior drives paid conversion and renewal risk. We turn event sequences into user-level commercial touchpoint features.


In [7]:
query = """-- build subscription_features
SELECT
    user_id,
    Max(case when event_type = 'trial_exposed' then 1 else 0 end) as trial_exposed_30d,
    Max(case when event_type = 'trial_started' then 1 else 0 end) as trial_started_30d,
    Max(case when event_type = 'paid_started' then 1 else 0 end) as paid_started_in_observation_30d,
    Sum(case when event_type = 'renewal_success' then 1 else 0 end) as renewal_success_count_30d,
    Sum(case when event_type = 'payment_failed' then 1 else 0 end) as payment_failed_count_30d,
    Sum(case when event_type = 'cancel' then 1 else 0 end) as cancel_count_30d,
    Round(Sum(price_usd),2) as subscription_revenue_observation_30d
FROM subscription_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 10
;"""

sql(query)


,user_id,trial_exposed_30d,trial_started_30d,paid_started_in_observation_30d,renewal_success_count_30d,payment_failed_count_30d,cancel_count_30d,subscription_revenue_observation_30d
0,U000001,0,0,0,0,0,1,10.99
1,U000002,1,0,0,0,0,0,0.00
2,U000006,1,0,0,0,0,0,0.00
3,U000007,1,0,0,0,0,0,0.00
4,U000009,1,0,1,0,0,0,10.99
5,U000010,0,0,0,1,0,0,10.99
6,U000015,1,1,0,0,0,0,0.00
7,U000016,1,0,0,1,0,0,10.99
8,U000017,1,0,1,0,0,0,10.99
9,U000020,0,0,0,1,0,0,16.99


## 7. Labels: Outcomes from the Prediction Window

The model predicts whether the user will churn or convert to paid in the future. Labels must come from the prediction window after the snapshot.


In [8]:
query = """
WITH churn_label AS (
    SELECT
        u.user_id,
        CASE
            -- If there are no listening events within 14 days after the snapshot (2026-04-01 to 2026-04-15), label as churned (1)
            WHEN COUNT(l.user_id) = 0 THEN 1
            ELSE 0
        END AS churn_label_14d
    FROM users u
    LEFT JOIN listening_events l
        ON u.user_id = l.user_id
       AND l.event_timestamp >= '2026-04-01'
       AND l.event_timestamp < '2026-04-15'
    GROUP BY u.user_id
),
conversion_label AS (
    SELECT
        u.user_id,
        CASE
            -- If there is a paid_started event within 30 days after the snapshot (2026-04-01 to 2026-05-01), label as converted (1)
            WHEN COUNT(s.user_id) > 0 THEN 1
            ELSE 0
        END AS paid_conversion_30d
    FROM users u
    LEFT JOIN subscription_events s
        ON u.user_id = s.user_id
       AND s.event_timestamp >= '2026-04-01'
       AND s.event_timestamp < '2026-05-01'
       AND s.event_type = 'paid_started'
    GROUP BY u.user_id
)
SELECT
    c.user_id,
    c.churn_label_14d,
    p.paid_conversion_30d
FROM churn_label c
LEFT JOIN conversion_label p
    ON c.user_id = p.user_id
LIMIT 10 ;
"""
sql(query)

,user_id,churn_label_14d,paid_conversion_30d
0,U000001,0,0
1,U000002,1,0
2,U000003,1,0
3,U000004,0,0
4,U000005,0,0
5,U000006,0,0
6,U000007,0,0
7,U000008,1,0
8,U000009,0,0
9,U000010,0,0


## 8. Assembling the User-Level Wide Table

Diagnostic tables only answer local questions; models and systematic EDA need a complete data asset with one row per user.


In [9]:
# The wide-table definition is versioned in sql/build_user_feature_table.sql
# (single source of truth — also used by scripts/build_user_feature_table.py).
ddl = (REPO_ROOT / "sql" / "build_user_feature_table.sql").read_text()
print(ddl)

run_script(ddl)
sql("SELECT * FROM user_level_feature_table LIMIT 5")

-- Builds the user-level feature/label wide table (view: user_level_feature_table).
-- Windows: observation 2026-03-02 .. 2026-04-01 (exclusive), snapshot 2026-04-01,
-- churn label window 14d (to 2026-04-15), conversion label window 30d (to 2026-05-01).
-- Executed against the in-memory SQLite database built by src/data_loader.py.

DROP VIEW IF EXISTS user_level_feature_table;

CREATE TEMP VIEW user_level_feature_table AS
WITH latest_subscription AS (
    SELECT user_id, plan_type AS latest_plan_type
    FROM (
        SELECT user_id, plan_type,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_timestamp DESC) AS rn
        FROM subscription_events
        WHERE event_timestamp < '2026-04-01'
    ) WHERE rn = 1
),
base_users AS (
    SELECT
        u.*,
        COALESCE(l.latest_plan_type, 'free') AS current_subscription_type,
        '2026-04-01' AS snapshot_date
    FROM users u
    LEFT JOIN latest_subscription l ON u.user_id = l.user_id
),
listening_features AS

,user_id,signup_date,country,language,region_tier,age_group,primary_device,acquisition_channel,student_eligible,marketing_opt_in,music_persona,current_subscription_type,snapshot_date,active_days_30d,listen_events_30d,listen_min_30d,sessions_30d,avg_play_duration_sec_30d,skip_rate_30d,completion_rate_30d,liked_songs_30d,playlist_adds_30d,search_events_30d,ad_impressions_30d,ad_clicks_30d,ad_click_rate_30d,ad_completions_30d,ad_completion_rate_30d,ad_revenue_30d,trial_exposed_30d,trial_started_30d,paid_started_in_observation_30d,renewal_success_count_30d,payment_failed_count_30d,cancel_count_30d,subscription_revenue_observation_30d,churn_label_14d,paid_conversion_30d
0,U000001,2026-02-06,UK,en,Tier 1,35-44,mobile_android,organic_search,0,0,commute_listener,individual_premium,2026-04-01,1,1,1.93,1,116.0,0.00,0.0,0,0,0,3,0,0.0,2,0.6667,0.0438,0,0,0,0,0,1,10.99,0,0
1,U000002,2026-03-07,ES,es,Tier 2,25-34,mobile_android,paid_social,0,0,playlist_builder,trial,2026-04-01,10,10,21.55,10,129.3,0.20,0.3,1,1,2,7,0,0.0,6,0.8571,0.0544,1,0,0,0,0,0,0.00,1,0
2,U000003,2025-07-29,IN,en,Tier 2,18-24,mobile_android,referral,0,0,casual_listener,trial,2026-04-01,3,3,6.50,3,130.0,0.00,0.0,0,0,0,1,0,0.0,0,0.0000,0.0025,0,0,0,0,0,0,0.00,1,0
3,U000004,2025-11-14,FR,fr,Tier 1,35-44,mobile_android,referral,0,1,commute_listener,trial,2026-04-01,4,4,11.33,4,170.0,0.25,0.0,0,0,3,2,0,0.0,1,0.5000,0.0703,0,0,0,0,0,0,0.00,0,0
4,U000005,2025-05-21,US,en,Tier 1,25-34,mobile_ios,student_partner,1,0,casual_listener,free,2026-04-01,1,1,2.72,1,163.0,0.00,0.0,0,0,0,0,0,0.0,0,0.0000,0.0000,0,0,0,0,0,0,0.00,0,0


## 9. QA: JOIN Inflation Check

The most dangerous wide-table problem is duplicated users. If row_count exceeds user_count, all downstream models and metrics will be wrong.


In [10]:
query = """-- QA 1: check the table is one row per user
SELECT
    COUNT(*) AS row_count,
    -- user_count
    -- duplicated_rows
    COUNT(DISTINCT user_id) AS user_count,
    (COUNT(*) - COUNT(DISTINCT user_id)) AS duplicated_rows
FROM user_level_feature_table
;"""

sql(query)

,row_count,user_count,duplicated_rows
0,50000,50000,0


## 10. QA: Label and Feature Distributions

Even if the wide table has no duplicates, we still need to check whether the labels and core features look reasonable.


In [11]:
query = """-- QA 2: check that core label and feature distributions look reasonable
SELECT
    -- churn_rate_14d
    -- paid_conversion_rate_30d
    -- avg_active_days_30d
    -- avg_listen_minutes_30d
    -- avg_ads_per_active_day

    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_min_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(ad_impressions_30d * 1.0 / NULLIF(active_days_30d, 0)), 2) AS avg_ads_per_active_day
FROM user_level_feature_table
;"""

sql(query)

,churn_rate_14d,paid_conversion_rate_30d,avg_active_days_30d,avg_listen_minutes_30d,avg_ads_per_active_day
0,0.4756,0.0,6.23,37.32,0.56


## 11. Exporting the Wide Table

The final deliverable of the SQL lessons is not throwaway queries, but a reusable data asset.


In [12]:
# Same export is available from the command line:
#   python scripts/build_user_feature_table.py
wide_df = sql("SELECT * FROM user_level_feature_table")
output_path = REPO_ROOT / "data" / "user_feature_table.csv"
wide_df.to_csv(output_path, index=False)
print(f"Exported {len(wide_df):,} rows to {output_path}")
wide_df.head()

Exported 50,000 rows to /Users/jackyjiang/Desktop/spotify-user-behavior-analytics/data/user_feature_table.csv


,user_id,signup_date,country,language,region_tier,age_group,primary_device,acquisition_channel,student_eligible,marketing_opt_in,music_persona,current_subscription_type,snapshot_date,active_days_30d,listen_events_30d,listen_min_30d,sessions_30d,avg_play_duration_sec_30d,skip_rate_30d,completion_rate_30d,liked_songs_30d,playlist_adds_30d,search_events_30d,ad_impressions_30d,ad_clicks_30d,ad_click_rate_30d,ad_completions_30d,ad_completion_rate_30d,ad_revenue_30d,trial_exposed_30d,trial_started_30d,paid_started_in_observation_30d,renewal_success_count_30d,payment_failed_count_30d,cancel_count_30d,subscription_revenue_observation_30d,churn_label_14d,paid_conversion_30d
0,U000001,2026-02-06,UK,en,Tier 1,35-44,mobile_android,organic_search,0,0,commute_listener,individual_premium,2026-04-01,1,1,1.93,1,116.0,0.00,0.0,0,0,0,3,0,0.0,2,0.6667,0.0438,0,0,0,0,0,1,10.99,0,0
1,U000002,2026-03-07,ES,es,Tier 2,25-34,mobile_android,paid_social,0,0,playlist_builder,trial,2026-04-01,10,10,21.55,10,129.3,0.20,0.3,1,1,2,7,0,0.0,6,0.8571,0.0544,1,0,0,0,0,0,0.00,1,0
2,U000003,2025-07-29,IN,en,Tier 2,18-24,mobile_android,referral,0,0,casual_listener,trial,2026-04-01,3,3,6.50,3,130.0,0.00,0.0,0,0,0,1,0,0.0,0,0.0000,0.0025,0,0,0,0,0,0,0.00,1,0
3,U000004,2025-11-14,FR,fr,Tier 1,35-44,mobile_android,referral,0,1,commute_listener,trial,2026-04-01,4,4,11.33,4,170.0,0.25,0.0,0,0,3,2,0,0.0,1,0.5000,0.0703,0,0,0,0,0,0,0.00,0,0
4,U000005,2025-05-21,US,en,Tier 1,25-34,mobile_ios,student_partner,1,0,casual_listener,free,2026-04-01,1,1,2.72,1,163.0,0.00,0.0,0,0,0,0,0,0.0,0,0.0000,0.0000,0,0,0,0,0,0,0.00,0,0
